## 1. Environment Setup and Artifact Loading

In [1]:
from __future__ import annotations

import json
import os
import pickle
import time
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
import shap
import torch

from lib.explainability import build_transaction_ae_features, load_temporal_autoencoder
from lib.model_utils import load_artifacts, load_sage_model
from lib.resource_paths import resolve_output_path

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.explain import Explainer, GNNExplainer
    from torch_geometric.loader import NeighborLoader
    import torch_geometric.transforms as T
    TORCH_GEO_OK = True
except Exception:
    TORCH_GEO_OK = False

OUTPUTS_DIR = Path(resolve_output_path())
paths = {
    'rank_df': OUTPUTS_DIR / 'rank_df_with_anchor_expansion.csv.gz',
    'model_output': OUTPUTS_DIR / 'model_output.csv',
    'sage_model': OUTPUTS_DIR / 'fraud_sage_model.pth',
    'sage_artifacts': OUTPUTS_DIR / 'sage_artifacts.pkl',
    'lgbm': OUTPUTS_DIR / 'lgbm_fraud_ranker.joblib',
    'ae_scores': OUTPUTS_DIR / 'autoencoder_transaction_scores.csv.gz',
    'master_pool': OUTPUTS_DIR / 'master_transaction_pool.csv.gz',
}

missing = [k for k, p in paths.items() if not p.exists()]
assert not missing, f'Missing required artifacts: {missing}'

rank_df = pd.read_csv(paths['rank_df'])
model_output = pd.read_csv(paths['model_output'])
ae_scores = pd.read_csv(paths['ae_scores'])
master_pool = pd.read_csv(paths['master_pool'], compression='gzip', low_memory=False)

rank_df['customer_id'] = rank_df['customer_id'].astype(str)
model_output['customer_id'] = model_output['customer_id'].astype(str)
ae_scores['customer_id'] = ae_scores['customer_id'].astype(str)
master_pool['customer_id'] = master_pool['customer_id'].astype(str)

print('Artifacts loaded from:', OUTPUTS_DIR)
print('Rows -> rank_df:', len(rank_df), 'model_output:', len(model_output), 'master_pool:', len(master_pool))

Artifacts loaded from: /student/minalex/imi-bigdata-2026/webapp_resources/outputs
Rows -> rank_df: 61410 model_output: 61410 master_pool: 5903333


## 2. Select 3 Customers by Risk Band (Low / Medium / High)

In [2]:
risk_col = 'fraud_score' if 'fraud_score' in model_output.columns else 'scarcity_anchor_ensemble_prob'
scores = pd.to_numeric(model_output[risk_col], errors='coerce').fillna(0.0).clip(0.0, 1.0)
df = model_output[['customer_id']].copy()
df['risk_score'] = scores

low_df = df[df['risk_score'] <= 0.33].sort_values(['risk_score', 'customer_id'])
mid_df = df[(df['risk_score'] > 0.33) & (df['risk_score'] <= 0.66)].sort_values(['risk_score', 'customer_id'])
high_df = df[df['risk_score'] > 0.66].sort_values(['risk_score', 'customer_id'], ascending=[False, True])

assert len(low_df) > 0 and len(mid_df) > 0 and len(high_df) > 0, 'Could not find all risk bands.'

selected = pd.concat([
    low_df.head(1).assign(risk_band='low'),
    mid_df.head(1).assign(risk_band='medium'),
    high_df.head(1).assign(risk_band='high'),
], ignore_index=True)

selected_ids = selected['customer_id'].tolist()
selected_payload = selected.to_dict(orient='records')
sel_path = OUTPUTS_DIR / 'explainability_selected_customers.json'
sel_path.write_text(json.dumps(selected_payload, indent=2))
print('Saved:', sel_path)
display(selected)

Saved: /student/minalex/imi-bigdata-2026/webapp_resources/outputs/explainability_selected_customers.json


,customer_id,risk_score,risk_band
0,SYNID0106469074,0.097329,low
1,SYNID0109559694,0.330047,medium
2,SYNID0107746512,0.832079,high


## 3. Test LLM Endpoint Connectivity and Response Format

In [3]:
OLLAMA_URL = os.environ.get('OLLAMA_URL', 'http://132.145.111.57:11434')
start_t = time.time()
llm_test = {'url': OLLAMA_URL, 'success': False, 'model': None, 'latency_sec': None, 'error': None, 'sample': None}

try:
    tags_r = requests.get(f'{OLLAMA_URL}/api/tags', timeout=8)
    tags_r.raise_for_status()
    models = [m.get('name') for m in tags_r.json().get('models', []) if m.get('name')]
    assert models, 'No models found on endpoint'
    model_name = models[0]
    gen_r = requests.post(
        f'{OLLAMA_URL}/api/generate',
        json={'model': model_name, 'prompt': 'Reply with: ok', 'stream': False, 'options': {'temperature': 0.0, 'num_predict': 20}},
        timeout=20,
    )
    gen_r.raise_for_status()
    out = gen_r.json()
    llm_test['success'] = isinstance(out, dict) and ('response' in out)
    llm_test['model'] = model_name
    llm_test['sample'] = str(out.get('response', ''))[:200]
except Exception as e:
    llm_test['error'] = str(e)

llm_test['latency_sec'] = round(time.time() - start_t, 3)
llm_test_path = OUTPUTS_DIR / 'llm_endpoint_test.json'
llm_test_path.write_text(json.dumps(llm_test, indent=2))
print('Saved:', llm_test_path)
print(json.dumps(llm_test, indent=2))

Saved: /student/minalex/imi-bigdata-2026/webapp_resources/outputs/llm_endpoint_test.json
{
  "url": "http://132.145.111.57:11434",
  "success": true,
  "model": "gemma2:2b",
  "latency_sec": 0.619,
  "error": null,
  "sample": "ok \n"
}


## 4. Run GNNExplainer for Selected Customers and Save Web-App-Friendly Output

In [4]:
gnn_jsonl_path = OUTPUTS_DIR / 'gnn_explainer_webapp.jsonl'
gnn_records = []
band_map = dict(zip(selected['customer_id'], selected['risk_band']))

if TORCH_GEO_OK:
    arts = load_artifacts()
    model_sage = load_sage_model(artifacts=arts)

    data = HeteroData()
    data['customer'].x = arts['x_cust']
    data['category'].x = arts['x_cat']
    data['city'].x = arts['x_city']
    data[('customer', 'purchases_at', 'category')].edge_index = arts['edge_cust_cat']
    data[('customer', 'transacts_in', 'city')].edge_index = arts['edge_cust_city']
    data = T.ToUndirected()(data)

    cust_map = arts['cust_map']
    rev_cust = {v: k for k, v in cust_map.items()}

    class Wrap(torch.nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base = base
        def forward(self, x_dict, edge_index_dict):
            return self.base(x_dict, edge_index_dict).squeeze(-1)

    explainer = Explainer(
        model=Wrap(model_sage).eval(),
        algorithm=GNNExplainer(epochs=15),
        explanation_type='model',
        node_mask_type='attributes',
        edge_mask_type='object',
        model_config=dict(mode='binary_classification', task_level='node', return_type='raw'),
    )

    for cid in selected_ids:
        rec = {'customer_id': cid, 'risk_band': band_map[cid], 'important_edges': [], 'important_nodes': [], 'mask_summary': {}, 'runtime_sec': None}
        t0 = time.time()
        try:
            idx = int(cust_map[cid])
            seed = torch.tensor([idx], dtype=torch.long)
            sub = next(iter(NeighborLoader(data, num_neighbors=[20, 15, 10], input_nodes=('customer', seed), batch_size=1, shuffle=False, num_workers=0)))
            loc = int(torch.where(sub['customer'].n_id == idx)[0][0].item())
            exp = explainer(sub.x_dict, sub.edge_index_dict, index=loc)

            for rel, mask in getattr(exp, 'edge_mask_dict', {}).items():
                if mask is None or mask.numel() == 0:
                    continue
                topk = min(5, int(mask.numel()))
                vals, pos = torch.topk(mask, k=topk)
                eidx = sub.edge_index_dict[rel]
                rel_name = '__'.join(rel)
                for v, p in zip(vals.tolist(), pos.tolist()):
                    src = int(eidx[0, p].item())
                    dst = int(eidx[1, p].item())
                    rec['important_edges'].append({'relation': rel_name, 'score': float(v), 'src_local': src, 'dst_local': dst})

            cm = getattr(exp, 'node_mask_dict', {}).get('customer')
            if cm is not None and cm.ndim == 2 and loc < cm.shape[0]:
                m = cm[loc]
                topf = torch.topk(m, k=min(5, m.numel()))
                rec['important_nodes'] = [{'feature_index': int(i), 'score': float(s)} for s, i in zip(topf.values.tolist(), topf.indices.tolist())]

            scores = [e['score'] for e in rec['important_edges']]
            rec['mask_summary'] = {
                'edge_count': len(scores),
                'edge_mean': float(np.mean(scores)) if scores else 0.0,
                'edge_max': float(np.max(scores)) if scores else 0.0,
            }
        except Exception as e:
            rec['mask_summary'] = {'error': str(e)}

        rec['runtime_sec'] = round(time.time() - t0, 3)
        gnn_records.append(rec)
else:
    for cid in selected_ids:
        gnn_records.append({
            'customer_id': cid,
            'risk_band': band_map[cid],
            'important_edges': [],
            'important_nodes': [],
            'mask_summary': {'error': 'torch_geometric unavailable'},
            'runtime_sec': 0.0,
        })

with open(gnn_jsonl_path, 'w', encoding='utf-8') as f:
    for rec in gnn_records:
        f.write(json.dumps(rec) + '\n')

print('Saved:', gnn_jsonl_path)
print('Records:', len(gnn_records))

Saved: /student/minalex/imi-bigdata-2026/webapp_resources/outputs/gnn_explainer_webapp.jsonl
Records: 3


## 5. Compute SHAP Values for LightGBM and Save Top Feature Attributions

In [5]:
# ── Section 5: SHAP Values for LightGBM ──────────────────────────────────────
# The LGB model was trained with .values (no column names → Column_0..32).
# We feed it rank_df which contains all original training features.

shap_jsonl_path = OUTPUTS_DIR / 'lgbm_shap_webapp.jsonl'
lgb_model = joblib.load(paths['lgbm'])

LGB_FEATURES = [
    'emb_pca_1','emb_pca_2','emb_pca_3','emb_pca_4',
    'emb_pca_5','emb_pca_6','emb_pca_7','emb_pca_8',
    'km_component_size','km_component_train_fraud_rate','km_component_mean_dgi',
    'hdb_component_size','hdb_component_fraud_rate_labeled','hdb_component_fraud_lift_labeled',
    'knn_mean_distance','knn_suspicious_share','knn_gold_fraud_count',
    'dist_to_fraud_centroid','dist_to_legit_centroid','centroid_margin',
    'dgi_anomaly_score','customer_ae_risk_norm','gmm_max_prob','component_confidence',
    'mlp_fraud_prob','cluster_consensus_score',
    'hdb_outlier_score','min_dist_to_fraud_anchor','mean_dist_to_fraud_anchor',
    'min_dist_to_legit_anchor','anchor_proximity_score',
    'eft_amount_match_count','abm_dc_colocated',
]
LGB_FEATURES = [c for c in LGB_FEATURES if c in rank_df.columns]
assert len(LGB_FEATURES) == 33, f"Expected 33 LGB features, got {len(LGB_FEATURES)}"

rdf = rank_df.set_index('customer_id')
X = rdf.reindex(selected_ids)[LGB_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0.0)

expl = shap.TreeExplainer(lgb_model)
shap_vals = expl.shap_values(X.values)
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]  # class-1 (fraud) SHAP values

shap_records = []
for i, cid in enumerate(selected_ids):
    sv = shap_vals[i]
    top_idx = np.argsort(np.abs(sv))[::-1][:10]
    top_feats = [
        {"feature": LGB_FEATURES[j], "shap_value": float(sv[j]), "feature_value": float(X.values[i, j])}
        for j in top_idx
    ]
    shap_records.append({
        "customer_id": cid,
        "risk_band": band_map[cid],
        "lgb_fraud_prob": float(lgb_model.predict_proba(X.values[i:i+1])[0, 1]),
        "base_value": float(expl.expected_value if not isinstance(expl.expected_value, (list, np.ndarray)) else expl.expected_value[1]),
        "top_shap_features": top_feats,
    })

with open(shap_jsonl_path, 'w', encoding='utf-8') as f:
    for r in shap_records:
        f.write(json.dumps(r) + '\n')

print(f"Saved: {shap_jsonl_path}")
print(f"Records: {len(shap_records)}")
for r in shap_records:
    top = r['top_shap_features'][0]
    print(f"  {r['customer_id']} ({r['risk_band']}) lgb_prob={r['lgb_fraud_prob']:.3f}  top_feat={top['feature']}  shap={top['shap_value']:+.3f}")

Saved: /student/minalex/imi-bigdata-2026/webapp_resources/outputs/lgbm_shap_webapp.jsonl
Records: 3
  SYNID0106469074 (low) lgb_prob=0.069  top_feat=anchor_proximity_score  shap=-0.404
  SYNID0109559694 (medium) lgb_prob=0.072  top_feat=anchor_proximity_score  shap=-0.404
  SYNID0107746512 (high) lgb_prob=0.834  top_feat=anchor_proximity_score  shap=+3.031


/student/minalex/.local/lib/python3.10/site-packages/shap/explainers/_tree.py:586: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/student/minalex/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/student/minalex/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/student/minalex/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 6. Extract Autoencoder Anomalous Transactions for Selected Customers

In [6]:
ae_jsonl_path = OUTPUTS_DIR / 'ae_anomalies_webapp.jsonl'

tx = master_pool[master_pool['customer_id'].isin(selected_ids)].copy()
tx['transaction_datetime'] = pd.to_datetime(tx['transaction_datetime'], errors='coerce')

# Prefer saved per-transaction AE scores if transaction_id exists in both tables
if 'transaction_id' in tx.columns and 'transaction_id' in ae_scores.columns:
    ae_scores['transaction_id'] = ae_scores['transaction_id'].astype(str)
    tx['transaction_id'] = tx['transaction_id'].astype(str)
    tx = tx.merge(ae_scores[['transaction_id', 'customer_id', 'reconstruction_error', 'ae_risk_score']], on=['transaction_id', 'customer_id'], how='left')

if 'reconstruction_error' not in tx.columns:
    tx['reconstruction_error'] = np.nan
if 'ae_risk_score' not in tx.columns:
    tx['ae_risk_score'] = np.nan

ae_records = []
for cid in selected_ids:
    c = tx[tx['customer_id'] == cid].copy()
    c['reconstruction_error'] = pd.to_numeric(c['reconstruction_error'], errors='coerce')
    c['ae_risk_score'] = pd.to_numeric(c['ae_risk_score'], errors='coerce')
    c = c.sort_values(['ae_risk_score', 'reconstruction_error'], ascending=False).head(10)

    for _, r in c.iterrows():
        ae_records.append({
            'customer_id': cid,
            'risk_band': band_map[cid],
            'transaction_id': str(r.get('transaction_id', '')),
            'transaction_datetime': str(r.get('transaction_datetime', '')),
            'amount_cad': float(pd.to_numeric(r.get('amount_cad', 0.0), errors='coerce') or 0.0),
            'merchant_category': str(r.get('merchant_category', '')),
            'city': str(r.get('city', '')),
            'source_dataset': str(r.get('source_dataset', '')),
            'cash_indicator': int(pd.to_numeric(r.get('cash_indicator', 0), errors='coerce') or 0),
            'ecommerce_ind': int(pd.to_numeric(r.get('ecommerce_ind', 0), errors='coerce') or 0),
            'reconstruction_error': float(pd.to_numeric(r.get('reconstruction_error', 0.0), errors='coerce') or 0.0),
            'ae_risk_score': float(pd.to_numeric(r.get('ae_risk_score', 0.0), errors='coerce') or 0.0),
        })

with open(ae_jsonl_path, 'w', encoding='utf-8') as f:
    for rec in ae_records:
        f.write(json.dumps(rec) + '\n')

print('Saved:', ae_jsonl_path)
print('Rows:', len(ae_records))

Saved: /student/minalex/imi-bigdata-2026/webapp_resources/outputs/ae_anomalies_webapp.jsonl
Rows: 30


## 7. Batch LLM Explanations for GNN + SHAP + AE Evidence

In [7]:
# ── Section 7: Batch LLM Explanations (Business-Friendly + Behavior-First) ─────
explain_jsonl_path = OUTPUTS_DIR / 'customer_explanations_webapp.jsonl'

gnn_by_cid = {r['customer_id']: r for r in gnn_records}
shap_by_cid = {r['customer_id']: r for r in shap_records}
ae_by_cid = {}
for r in ae_records:
    ae_by_cid.setdefault(r['customer_id'], []).append(r)

RELATION_LABELS = {
    'customer__purchases_at__category': 'repeat spending in similar merchant categories',
    'customer__transacts_in__city': 'repeat activity concentrated in specific cities',
    'category__rev_purchases_at__customer': 'shared merchant-category behavior with peer customers',
    'city__rev_transacts_in__customer': 'shared location behavior with peer customers',
}

FEATURE_LABELS = {
    'anchor_proximity_score': 'similarity to previously flagged customer behavior',
    'min_dist_to_fraud_anchor': 'distance to suspicious behavior reference groups',
    'min_dist_to_legit_anchor': 'distance to typical legitimate behavior groups',
    'mlp_fraud_prob': 'overall pattern-risk score from behavior profile',
    'component_confidence': 'stability/consistency of behavior cluster assignment',
    'knn_suspicious_share': 'share of nearest behavioral peers previously flagged',
    'knn_gold_fraud_count': 'count of nearest peers with confirmed adverse outcomes',
    'dist_to_fraud_centroid': 'distance to suspicious behavior center',
    'dist_to_legit_centroid': 'distance to legitimate behavior center',
}

def _feature_lookup(shap_record):
    lookup = {}
    for item in shap_record.get('top_shap_features', []):
        name = item.get('feature')
        if name:
            lookup[name] = item
    return lookup

def describe_anchor_behavior(shap_record):
    feats = _feature_lookup(shap_record)
    prox_item = feats.get('anchor_proximity_score', {})
    prox_val = float(prox_item.get('feature_value', 0.0) or 0.0)
    prox_impact = float(prox_item.get('shap_value', 0.0) or 0.0)
    dist_fraud = float(feats.get('min_dist_to_fraud_anchor', {}).get('feature_value', np.nan))
    dist_legit = float(feats.get('min_dist_to_legit_anchor', {}).get('feature_value', np.nan))

    if prox_val >= 0.85 or prox_impact >= 0.8:
        lead = 'Behavior is strongly similar to previously flagged high-risk customer profiles.'
        strength = 'strong'
    elif prox_val >= 0.55 or prox_impact >= 0.2:
        lead = 'Behavior has partial overlap with previously flagged customer profiles.'
        strength = 'moderate'
    else:
        lead = 'Behavior does not show strong similarity to previously flagged profiles.'
        strength = 'weak'

    if np.isfinite(dist_fraud) and np.isfinite(dist_legit):
        if dist_fraud < dist_legit:
            detail = 'The profile sits closer to suspicious reference behavior than to typical legitimate behavior.'
        else:
            detail = 'The profile sits closer to legitimate reference behavior than to suspicious reference behavior.'
    else:
        detail = 'Reference-group distance signals are limited for this customer.'

    return {
        'text': f"{lead} {detail}",
        'strength': strength,
        'prox_value': prox_val,
        'prox_impact': prox_impact,
        'dist_fraud': dist_fraud,
        'dist_legit': dist_legit,
    }

def describe_network_behavior(gnn_record):
    edges = gnn_record.get('important_edges', [])[:3]
    if not edges:
        return {
            'text': 'No dominant relationship concentration was surfaced in this run.',
            'strength': 'weak',
        }

    labels = []
    max_score = 0.0
    for e in edges:
        rel = str(e.get('relation', ''))
        s = float(e.get('score', 0.0) or 0.0)
        max_score = max(max_score, s)
        labels.append(f"{RELATION_LABELS.get(rel, rel)} ({s:.2f})")

    strength = 'strong' if max_score >= 0.65 else ('moderate' if max_score >= 0.45 else 'weak')
    return {
        'text': 'Relationship patterns show: ' + '; '.join(labels) + '.',
        'strength': strength,
    }

def describe_anomaly_behavior(ae_items):
    if not ae_items:
        return {
            'text': 'No major transaction-level anomaly was surfaced for this customer in the sampled records.',
            'strength': 'weak',
            'top_amount': 0.0,
            'top_source': '',
        }

    top = ae_items[0]
    merchant = str(top.get('merchant_category', '?'))
    city = str(top.get('city', '?'))
    amount = float(top.get('amount_cad', 0.0) or 0.0)
    score = float(top.get('ae_risk_score', top.get('reconstruction_error', 0.0)) or 0.0)
    source = str(top.get('source_dataset', ''))

    strength = 'strong' if score >= 0.95 else ('moderate' if score >= 0.7 else 'weak')
    return {
        'text': (
            f"Anomalies were detected, with the strongest event in {merchant} ({city}) for ${amount:,.2f} "
            f"and anomaly intensity {score:.4f}."
        ),
        'strength': strength,
        'top_amount': amount,
        'top_source': source,
    }

def summarize_top_drivers(shap_record):
    labels = []
    for item in shap_record.get('top_shap_features', [])[:4]:
        f = str(item.get('feature', ''))
        labels.append(FEATURE_LABELS.get(f, f.replace('_', ' ')))
    return '; '.join(labels) if labels else 'No clear dominant driver list was available.'

def infer_pattern_hints(anchor_info, network_info, anomaly_info):
    hints = []
    if anomaly_info['top_source'] == 'scotia_eft' and anomaly_info['top_amount'] >= 1000:
        hints.append('Large transfer movement should be validated against known account purpose and expected cash flow.')
    if network_info['strength'] in {'strong', 'moderate'}:
        hints.append('Repeated category/location concentration may indicate coordinated behavior or repeated routines.')
    if anchor_info['strength'] in {'strong', 'moderate'}:
        hints.append('Similarity to previously flagged populations suggests this case deserves closer review.')
    if not hints:
        hints.append('No single pattern dominates; prioritize direct review of timeline, counterparties, and transaction context.')
    return ' '.join(hints)

def consistency_note(risk_band, pred, anchor_info, anomaly_info, network_info):
    evidence_rank = {'weak': 0, 'moderate': 1, 'strong': 2}
    evidence = max(evidence_rank[anchor_info['strength']], evidence_rank[anomaly_info['strength']], evidence_rank[network_info['strength']])

    if risk_band == 'low' and evidence <= 1:
        return 'Important: avoid overstating concern; describe this as lower-priority with targeted checks.'
    if risk_band in {'medium', 'high'} and anchor_info['strength'] == 'weak' and evidence >= 1:
        return 'Important: if profile similarity is weak, explain that risk is driven mainly by transaction/network factors, not profile similarity.'
    if pred >= 0.75 and evidence == 0:
        return 'Important: explain model confidence while clearly noting weak observable evidence in current slice.'
    return 'Keep tone balanced: evidence-led, no overstatement.'

def needs_rewrite(text, risk_band, anchor_info, anomaly_info):
    t = text.lower()
    blocked_terms = [' shap', ' gnn', ' ae ', 'embedding', 'graph model', 'customer__', 'typology', 'reduces risk', 'impact +', 'impact -']
    if any(term in t for term in blocked_terms):
        return True
    if 'reducing risk but also raises concerns' in t:
        return True
    if risk_band == 'low' and ('high-risk' in t or 'high risk' in t):
        return True
    if anchor_info['strength'] == 'weak' and ('strongly similar' in t or 'closely mirrors' in t):
        return True
    if 'closer to legitimate' in t and risk_band in {'medium', 'high'} and anomaly_info['strength'] in {'moderate', 'strong'} and 'driven by' not in t:
        return True
    return False

def rewrite_with_constraints(text, cid, band, pred, driver_text, anchor_info, network_info, anomaly_info, pattern_hints):
    rewrite_prompt = (
        'Rewrite the case note to remove contradictions and keep it business-friendly.\n'
        'Rules: one paragraph, 4-6 sentences, plain English, no technical acronyms, no contradictions.\n'
        'Do not mention typology labels. Focus on observed behavior and practical checks.\n'
        'Do not include signed model contribution language (for example impact +/- or reduces risk).\n'
        'If profile similarity is weak, say so clearly and explain what else drove the flag.\n'
        'Do not treat low-risk-like behavior as concerning by itself.\n\n'
        f'Customer ID: {cid}\nRisk band: {band}\nEstimated fraud probability: {pred:.4f}\n'
        f'Driver summary: {driver_text}\n'
        f'Anchor insight: {anchor_info["text"]}\n'
        f'Network insight: {network_info["text"]}\n'
        f'Anomaly insight: {anomaly_info["text"]}\n'
        f'Pattern hints: {pattern_hints}\n\n'
        f'Original note to fix:\n{text}'
    )
    revised = call_llm(rewrite_prompt)
    return revised if revised else text

def fallback_text(cid, band, pred, anchor_text, network_text, anomaly_text, driver_text, pattern_hints):
    return (
        f"Customer {cid} is currently rated {band} risk with model-estimated probability {pred:.3f}. "
        f"Key factors reviewed were {driver_text}. {anchor_text} {network_text} {anomaly_text} "
        f"Priority checks: {pattern_hints}"
    )

def call_llm(prompt):
    if not llm_test.get('success'):
        raise RuntimeError('endpoint unavailable')
    resp = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            'model': llm_test['model'],
            'prompt': prompt,
            'stream': False,
            'options': {'temperature': 0.25, 'num_predict': 280},
        },
        timeout=45,
    )
    resp.raise_for_status()
    return str(resp.json().get('response', '')).strip()

explanation_records = []
for cid in selected_ids:
    band = band_map[cid]
    s = shap_by_cid.get(cid, {})
    g = gnn_by_cid.get(cid, {})
    a = ae_by_cid.get(cid, [])

    pred = float(s.get('lgb_fraud_prob', s.get('prediction', 0.0)))
    anchor_info = describe_anchor_behavior(s)
    network_info = describe_network_behavior(g)
    anomaly_info = describe_anomaly_behavior(a)
    driver_text = summarize_top_drivers(s)
    pattern_hints = infer_pattern_hints(anchor_info, network_info, anomaly_info)
    guidance = consistency_note(band, pred, anchor_info, anomaly_info, network_info)

    prompt = (
        'Write a concise investigator case note for business professionals reviewing suspicious financial activity.\n'
        'Use plain English; do not use technical model terms or acronyms.\n'
        'Do not use words like SHAP, GNN, AE, feature attribution, embeddings, node, graph model, centroid, or kNN.\n'
        'Keep it to one short paragraph (4-6 sentences), no bullet points.\n'
        'Do not mention typology names unless there is direct, concrete evidence.\n'
        'Prefer concrete factors such as anomaly size, unusual timing, concentration patterns, and implausible movement over labels.\n'
        'Do not include signed model contribution language (for example impact +/- or reduces risk).\n'
        'If a clear pattern label is not supported, simply explain factors and why the model flagged the case.\n'
        'Never force a pattern label. Avoid contradictions (for example, do not call low-risk similarity concerning).\n'
        'Explain what most likely drove the flag, what weakens confidence, and what the investigator should verify next.\n\n'
        f'Customer ID: {cid}\n'
        f'Risk band: {band}\n'
        f'Estimated fraud probability: {pred:.4f}\n'
        f'Key driver themes: {driver_text}\n'
        f'Behavior similarity insight: {anchor_info["text"]}\n'
        f'Relationship-pattern insight: {network_info["text"]}\n'
        f'Transaction anomaly insight: {anomaly_info["text"]}\n'
        f'Pattern hints from current evidence: {pattern_hints}\n'
        f'Consistency instruction: {guidance}\n'
    )

    llm_status = 'ok'
    explanation_text = ''
    for attempt in range(3):
        try:
            explanation_text = call_llm(prompt)
            if explanation_text:
                break
        except Exception as e:
            llm_status = f"error_attempt_{attempt + 1}: {e}"
            time.sleep(0.5 * (attempt + 1))

    if explanation_text and needs_rewrite(explanation_text, band, anchor_info, anomaly_info):
        try:
            explanation_text = rewrite_with_constraints(
                explanation_text, cid, band, pred, driver_text, anchor_info, network_info, anomaly_info, pattern_hints
            )
            llm_status = 'ok_rewritten'
        except Exception as e:
            llm_status = f'rewrite_error: {e}'

    if not explanation_text:
        llm_status = 'fallback'
        explanation_text = fallback_text(
            cid,
            band,
            pred,
            anchor_info['text'],
            network_info['text'],
            anomaly_info['text'],
            driver_text,
            pattern_hints,
        )

    explanation_records.append({
        'customer_id': cid,
        'risk_band': band,
        'model_fraud_prob': pred,
        'driver_summary': driver_text,
        'anchor_behavior_summary': anchor_info['text'],
        'network_behavior_summary': network_info['text'],
        'anomaly_summary': anomaly_info['text'],
        'pattern_hints': pattern_hints,
        'explanation_text': explanation_text,
        'llm_status': llm_status,
        'prompt_used': prompt,
    })

with open(explain_jsonl_path, 'w', encoding='utf-8') as f:
    for rec in explanation_records:
        f.write(json.dumps(rec) + '\n')

print(f"Saved: {explain_jsonl_path}")
print(f"Records: {len(explanation_records)}")
for rec in explanation_records:
    print(f"\n{'='*60}")
    print(f"Customer: {rec['customer_id']}  ({rec['risk_band']})  LLM: {rec['llm_status']}")
    print(rec['explanation_text'])

Saved: /student/minalex/imi-bigdata-2026/webapp_resources/outputs/customer_explanations_webapp.jsonl
Records: 3

Customer: SYNID0106469074  (low)  LLM: ok
Customer SYNID0106469074 has a low risk of fraud with an estimated probability of 0.0686.  The customer's transactions show some similarity to previously flagged customers in terms of spending patterns, but the profile is closer to legitimate reference behavior than suspicious behavior. The customer exhibits repeated spending in similar merchant categories, suggesting potential coordination or routine-based activity. However, there are anomalies detected with a strong event in 7011 (other) for $828.66, which warrants further investigation.  The investigator should verify if these unusual transactions represent isolated events or part of a larger pattern and examine the customer's spending history to understand the context behind these high-value transactions.

Customer: SYNID0109559694  (medium)  LLM: ok
Customer SYNID0109559694 exhi

## 8. Write Unified Web App Explanation Files and Verification Preview

In [8]:

# ── Section 8: Write Unified Web App Explanation Files ───────────────────────
bundle_jsonl_path = OUTPUTS_DIR / 'explainability_bundle_webapp.jsonl'
bundle_csv_path   = OUTPUTS_DIR / 'explainability_bundle_webapp.csv'

# Re-read all JSONL outputs to ensure consistent data
def _read_jsonl(path):
    with open(path) as fh:
        return [json.loads(line) for line in fh if line.strip()]

shap_data = {r['customer_id']: r for r in _read_jsonl(OUTPUTS_DIR / 'lgbm_shap_webapp.jsonl')}
ae_data   = {}
for r in _read_jsonl(OUTPUTS_DIR / 'ae_anomalies_webapp.jsonl'):
    ae_data.setdefault(r['customer_id'], []).append(r)
exp_data  = {r['customer_id']: r for r in _read_jsonl(OUTPUTS_DIR / 'customer_explanations_webapp.jsonl')}
gnn_data  = {r['customer_id']: r for r in _read_jsonl(OUTPUTS_DIR / 'gnn_explainer_webapp.jsonl')}

# Risk scores from the original model output
sel_scores = selected.set_index('customer_id')['risk_score'].to_dict()

rows = []
for cid in selected_ids:
    band = band_map[cid]

    s = shap_data.get(cid, {})
    risk_score = s.get('lgb_fraud_prob', sel_scores.get(cid, 0.0))
    top_shap_feats = s.get('top_shap_features', [])
    top_shap_name  = top_shap_feats[0]['feature'] if top_shap_feats else ''

    ae_list  = ae_data.get(cid, [])
    top_txn  = ae_list[0] if ae_list else {}

    c_exp = exp_data.get(cid, {})

    row = {
        'customer_id': cid,
        'risk_band': band,
        'risk_score': float(risk_score),
        'top_shap_feature': top_shap_name,
        'top_anomalous_transaction_id': str(top_txn.get('transaction_id', '')),
        'top_anomalous_amount_cad': float(top_txn.get('amount_cad', 0.0) or 0.0),
        'explanation_text': c_exp.get('explanation_text', ''),
        'llm_status': c_exp.get('llm_status', ''),
    }
    rows.append(row)

bundle_df = pd.DataFrame(rows)
bundle_df.to_csv(bundle_csv_path, index=False)

with open(bundle_jsonl_path, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r) + '\n')

print('Saved:', bundle_jsonl_path)
print('Saved:', bundle_csv_path)
print(f'\nVerification preview ({len(rows)} customers):')
display(bundle_df[['customer_id', 'risk_band', 'risk_score', 'top_shap_feature', 'top_anomalous_transaction_id', 'explanation_text']])


Saved: /student/minalex/imi-bigdata-2026/webapp_resources/outputs/explainability_bundle_webapp.jsonl
Saved: /student/minalex/imi-bigdata-2026/webapp_resources/outputs/explainability_bundle_webapp.csv

Verification preview (3 customers):


,customer_id,risk_band,risk_score,top_shap_feature,top_anomalous_transaction_id,explanation_text
0,SYNID0106469074,low,0.068581,anchor_proximity_score,CARD24110204563506,Customer SYNID0106469074 has a low risk of fra...
1,SYNID0109559694,medium,0.071973,anchor_proximity_score,EFT24110252005678,Customer SYNID0109559694 exhibits a medium-ris...
2,SYNID0107746512,high,0.833627,anchor_proximity_score,EFT24110364133013,Customer SYNID0107746512 exhibits high risk ba...


In [9]:
# ── Section 9: Full-population explanations for the web app ───────────────────
full_out_csv = OUTPUTS_DIR / 'model_output_explanations.csv'

# Runtime controls (all optional via env vars).
MAX_CUSTOMERS = int(os.environ.get('FULL_EXPLAIN_MAX_CUSTOMERS', '0'))  # 0 => all
BATCH_SIZE = int(os.environ.get('FULL_EXPLAIN_BATCH_SIZE', '1024'))
CHECKPOINT_EVERY = int(os.environ.get('FULL_EXPLAIN_CHECKPOINT_EVERY', '2000'))
MAX_LLM_CUSTOMERS = int(os.environ.get('FULL_EXPLAIN_MAX_LLM', '300'))
RESUME = os.environ.get('FULL_EXPLAIN_RESUME', '1') == '1'

risk_col = 'fraud_score' if 'fraud_score' in model_output.columns else 'scarcity_anchor_ensemble_prob'
all_df = model_output[['customer_id', risk_col]].copy()
all_df['customer_id'] = all_df['customer_id'].astype(str)
all_df['risk_score'] = pd.to_numeric(all_df[risk_col], errors='coerce').fillna(0.0).clip(0.0, 1.0)
all_df = all_df[['customer_id', 'risk_score']].drop_duplicates('customer_id')
all_df = all_df.sort_values(['risk_score', 'customer_id'], ascending=[False, True]).reset_index(drop=True)

if MAX_CUSTOMERS > 0:
    all_df = all_df.head(MAX_CUSTOMERS).copy()

# Resume support: keep already-finished customers.
existing_df = pd.DataFrame()
done_ids = set()
if RESUME and full_out_csv.exists():
    try:
        existing_df = pd.read_csv(full_out_csv)
        if 'customer_id' in existing_df.columns:
            done_ids = set(existing_df['customer_id'].astype(str).tolist())
    except Exception:
        existing_df = pd.DataFrame()
        done_ids = set()

pending_df = all_df[~all_df['customer_id'].isin(done_ids)].copy().reset_index(drop=True)
print(f'Target customers: {len(all_df):,} | already done: {len(done_ids):,} | pending: {len(pending_df):,}')

if pending_df.empty:
    print('No pending customers. Full output is already up to date.')
else:
    # Ensure model and features are ready.
    lgb_model = joblib.load(paths['lgbm'])
    LGB_FEATURES = [
        'emb_pca_1','emb_pca_2','emb_pca_3','emb_pca_4',
        'emb_pca_5','emb_pca_6','emb_pca_7','emb_pca_8',
        'km_component_size','km_component_train_fraud_rate','km_component_mean_dgi',
        'hdb_component_size','hdb_component_fraud_rate_labeled','hdb_component_fraud_lift_labeled',
        'knn_mean_distance','knn_suspicious_share','knn_gold_fraud_count',
        'dist_to_fraud_centroid','dist_to_legit_centroid','centroid_margin',
        'dgi_anomaly_score','customer_ae_risk_norm','gmm_max_prob','component_confidence',
        'mlp_fraud_prob','cluster_consensus_score',
        'hdb_outlier_score','min_dist_to_fraud_anchor','mean_dist_to_fraud_anchor',
        'min_dist_to_legit_anchor','anchor_proximity_score',
        'eft_amount_match_count','abm_dc_colocated',
    ]
    LGB_FEATURES = [c for c in LGB_FEATURES if c in rank_df.columns]
    assert len(LGB_FEATURES) > 0, 'No LGB features found in rank_df.'

    rdf = rank_df.set_index('customer_id')
    X_pending = rdf.reindex(pending_df['customer_id'])[LGB_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    booster = lgb_model.booster_

    # Build suspicious-transaction lookup: top-3 AE events per customer.
    tx_cols = ['customer_id', 'transaction_id', 'transaction_datetime', 'amount_cad', 'merchant_category', 'city', 'source_dataset']
    pool_small = master_pool[[c for c in tx_cols if c in master_pool.columns]].copy()
    for c in ['customer_id', 'transaction_id']:
        if c in pool_small.columns:
            pool_small[c] = pool_small[c].astype(str)
    ae_small = ae_scores[[c for c in ['customer_id', 'transaction_id', 'reconstruction_error', 'ae_risk_score'] if c in ae_scores.columns]].copy()
    for c in ['customer_id', 'transaction_id']:
        if c in ae_small.columns:
            ae_small[c] = ae_small[c].astype(str)
    for c in ['reconstruction_error', 'ae_risk_score']:
        if c not in ae_small.columns:
            ae_small[c] = np.nan
    ae_small['ae_risk_score'] = pd.to_numeric(ae_small['ae_risk_score'], errors='coerce').fillna(0.0)
    ae_small['reconstruction_error'] = pd.to_numeric(ae_small['reconstruction_error'], errors='coerce').fillna(0.0)

    if 'transaction_id' in ae_small.columns and 'transaction_id' in pool_small.columns:
        top_ae = (
            ae_small.sort_values(['ae_risk_score', 'reconstruction_error'], ascending=False)
            .groupby('customer_id', as_index=False)
            .head(3)
        )
        top_ae = top_ae.merge(pool_small, on=['customer_id', 'transaction_id'], how='left')
    else:
        top_ae = pd.DataFrame(columns=['customer_id'])

    top_ae_map = {}
    if not top_ae.empty:
        for cid, g in top_ae.groupby('customer_id'):
            recs = []
            for _, r in g.iterrows():
                recs.append({
                    'transaction_id': str(r.get('transaction_id', '')),
                    'transaction_datetime': str(r.get('transaction_datetime', '')),
                    'amount_cad': float(pd.to_numeric(r.get('amount_cad', 0.0), errors='coerce') or 0.0),
                    'merchant_category': str(r.get('merchant_category', '')),
                    'city': str(r.get('city', '')),
                    'source_dataset': str(r.get('source_dataset', '')),
                    'ae_risk_score': float(pd.to_numeric(r.get('ae_risk_score', 0.0), errors='coerce') or 0.0),
                    'reconstruction_error': float(pd.to_numeric(r.get('reconstruction_error', 0.0), errors='coerce') or 0.0),
                })
            top_ae_map[str(cid)] = recs

    def _driver_items(shap_vec):
        idx = np.argsort(np.abs(shap_vec))[::-1][:3]
        out = []
        for j in idx:
            out.append({
                'feature': LGB_FEATURES[int(j)],
                'shap_value': float(shap_vec[int(j)]),
            })
        return out

    def _business_expl(cid, risk_score, drivers, ae_events):
        dnames = [d['feature'].replace('_', ' ') for d in drivers]
        dtxt = ', '.join(dnames) if dnames else 'behavioral consistency checks'
        if ae_events:
            top = ae_events[0]
            atxt = f"Anomalies were detected, with the strongest event at ${top.get('amount_cad', 0.0):,.2f} in {top.get('merchant_category', 'unknown category')}."
        else:
            atxt = 'No large transaction-level anomaly was detected in the available sample.'

        tier = 'high' if risk_score > 0.7 else ('medium' if risk_score > 0.4 else 'low')
        return (
            f"Customer {cid} is currently assessed as {tier} risk based on overall behavior patterns. "
            f"The strongest factors behind this flag relate to {dtxt}. "
            f"{atxt} "
            "Recommended next checks are source of funds, counterparty consistency, and whether this pattern is a recent change versus normal history."
        )

    def _call_llm(prompt):
        if not llm_test.get('success'):
            raise RuntimeError('endpoint unavailable')
        resp = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                'model': llm_test['model'],
                'prompt': prompt,
                'stream': False,
                'options': {'temperature': 0.25, 'num_predict': 220},
            },
            timeout=40,
        )
        resp.raise_for_status()
        return str(resp.json().get('response', '')).strip()

    use_llm = bool(llm_test.get('success')) and MAX_LLM_CUSTOMERS > 0
    llm_budget_remaining = MAX_LLM_CUSTOMERS
    out_records = []
    processed = 0

    for start in range(0, len(pending_df), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(pending_df))
        chunk = pending_df.iloc[start:end].copy()
        Xc = X_pending.iloc[start:end]
        contrib = booster.predict(Xc.values, pred_contrib=True)
        # shape (n, n_features+1) — last col is the bias term
        shap_vals = contrib[:, :-1]

        for local_i, (_, row) in enumerate(chunk.iterrows()):
            cid = str(row['customer_id'])
            rs = float(row['risk_score'])
            sv = shap_vals[local_i]
            drivers = _driver_items(sv)
            ae_events = top_ae_map.get(cid, [])

            explanation = ''
            llm_status = 'template'
            prompt_used = ''

            if use_llm and llm_budget_remaining > 0:
                prompt_used = (
                    'Write one short case note for a business investigator in plain English. '
                    'No technical model terms. 4-5 sentences, one paragraph. '
                    'Focus on concrete factors and what to verify next.\n'
                    f"Customer: {cid}\nRisk score: {rs:.4f}\n"
                    f"Top driver themes: {', '.join(d['feature'] for d in drivers)}\n"
                    f"Top anomaly events: {json.dumps(ae_events[:2])}\n"
                )
                try:
                    explanation = _call_llm(prompt_used)
                    llm_status = 'ok'
                    llm_budget_remaining -= 1
                except Exception as e:
                    llm_status = f'template_after_llm_error: {e}'

            if not explanation:
                explanation = _business_expl(cid, rs, drivers, ae_events)

            d1 = drivers[0] if len(drivers) > 0 else {'feature': '', 'shap_value': 0.0}
            d2 = drivers[1] if len(drivers) > 1 else {'feature': '', 'shap_value': 0.0}
            d3 = drivers[2] if len(drivers) > 2 else {'feature': '', 'shap_value': 0.0}
            top_txn = ae_events[0] if ae_events else {}

            out_records.append({
                'customer_id': cid,
                'explanation': explanation,
                'narrative': explanation,
                'risk_score': rs,
                'driver_1': d1['feature'],
                'driver_1_shap': float(d1['shap_value']),
                'driver_2': d2['feature'],
                'driver_2_shap': float(d2['shap_value']),
                'driver_3': d3['feature'],
                'driver_3_shap': float(d3['shap_value']),
                'top_anomalous_transaction_id': str(top_txn.get('transaction_id', '')),
                'top_anomalous_amount_cad': float(pd.to_numeric(top_txn.get('amount_cad', 0.0), errors='coerce') or 0.0),
                'driver_heatmap_json': json.dumps([
                    {'feature': d['feature'], 'score': float(abs(d['shap_value']))} for d in drivers
                ]),
                'suspicious_transactions_json': json.dumps(ae_events),
                'llm_status': llm_status,
                'prompt_used': prompt_used,
            })

        processed += len(chunk)

        # Periodic checkpoint write so long jobs can resume safely.
        if processed % CHECKPOINT_EVERY == 0 or end == len(pending_df):
            out_df = pd.DataFrame(out_records)
            merged_df = pd.concat([existing_df, out_df], ignore_index=True) if not existing_df.empty else out_df
            merged_df = merged_df.drop_duplicates('customer_id', keep='last')
            merged_df.to_csv(full_out_csv, index=False)
            print(f'Checkpoint saved -> {full_out_csv} | processed pending: {processed:,}/{len(pending_df):,}')

    print(f'Completed. Saved full explanations to: {full_out_csv}')
    print(f'Total rows now: {len(pd.read_csv(full_out_csv)):,}')

Target customers: 61,410 | already done: 0 | pending: 61,410


Checkpoint saved -> /student/minalex/imi-bigdata-2026/webapp_resources/outputs/model_output_explanations.csv | processed pending: 61,410/61,410
Completed. Saved full explanations to: /student/minalex/imi-bigdata-2026/webapp_resources/outputs/model_output_explanations.csv


Total rows now: 61,410
